# Pipeline di punteggio da file Excel
Versione dinamica con grafico a fasce colorate e asse X basato sull'indice della finestra temporale.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.chart import LineChart, Reference
from openpyxl.chart.shapes import GraphicalProperties
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.drawing.line import LineProperties

plt.style.use('seaborn-v0_8-whitegrid')
INPUT_FILES = [
    "GT X Carrying rimontato temporalmente AGNESE.xlsx",
    "GT X Carrying rimontato temporalmente NICK.xlsx",
    "GT X Carrying rimontato temporalmente SARA.xlsx",
    "GT X Carrying rimontato temporalmente SILVIA 2.xlsx",
    "GT X Carrying rimontato temporalmente TEO 2.xlsx"
]
OUTPUT_DIR = Path('output_pipeline')
OUTPUT_DIR.mkdir(exist_ok=True)
WINDOW_SIZE = 4
STEP_SIZE = 2
COLOR_TO_SCORE = {'verde': 12.0, 'giallo': 37.0, 'arancione': 62.0, 'rosso': 87.5}
CONFIDENCE_BAND = {'verde': 12.0, 'giallo': 12.0, 'arancione': 12.0, 'rosso': 12.5}

def normalize_text(x):
    if pd.isna(x): return np.nan
    return str(x).strip().lower()

def find_column(df, candidates):
    normalized = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in normalized: return normalized[candidate.lower()]
    for candidate in candidates:
        for normalized_name, original_name in normalized.items():
            if candidate.lower() in normalized_name: return original_name
    return None

def load_single_file(path):
    xls = pd.ExcelFile(path)
    raw = pd.read_excel(path, sheet_name=xls.sheet_names[0], header=None)
    header_row = None
    for index in range(len(raw)):
        values = [str(value).strip().lower() for value in raw.iloc[index].tolist() if pd.notna(value)]
        if 'frame_n' in values and 'traffic_light' in values:
            header_row = index
            break
    if header_row is None: raise ValueError(f"Header non trovato nel file {path}.")
    df = pd.read_excel(path, sheet_name=xls.sheet_names[0], header=header_row)
    df.columns = [str(column).strip() for column in df.columns]
    frame_column = find_column(df, ['frame_n', 'frame', 'frame number', 'numero frame'])
    color_column = find_column(df, ['traffic_light', 'traffic light', 'color', 'colore'])
    if frame_column is None or color_column is None: raise ValueError(f"Colonne frame/color non trovate nel file {path}.")
    out = df[[frame_column, color_column]].copy()
    out.columns = ['frame_n', 'traffic_light']
    out['frame_n'] = pd.to_numeric(out['frame_n'], errors='coerce')
    out['traffic_light'] = out['traffic_light'].map(normalize_text)
    out = out.dropna(subset=['frame_n', 'traffic_light'])
    out = out[out['traffic_light'].isin(COLOR_TO_SCORE)]
    out['frame_n'] = out['frame_n'].astype(int)
    out['score'] = out['traffic_light'].map(COLOR_TO_SCORE)
    out['confidence_half_band'] = out['traffic_light'].map(CONFIDENCE_BAND)
    return out.sort_values('frame_n').reset_index(drop=True)

def aggregate_frames(file_paths):
    if not file_paths: raise ValueError('Inserire almeno un file Excel in INPUT_FILES.')
    rows = []
    for trial_index, path in enumerate(file_paths, start=1):
        data = load_single_file(path).copy()
        data['trial_id'] = f'prova_{trial_index}'
        data['source_file'] = Path(path).name
        rows.append(data)
    long_df = pd.concat(rows, ignore_index=True)
    trial_matrix = long_df.pivot_table(index='trial_id', columns='frame_n', values='score', aggfunc='mean')
    frame_pipeline = long_df.groupby('frame_n', as_index=False).agg(mean_frame_score=('score', 'mean'), std_frame_score=('score', 'std'), n_votes=('score', 'count')).sort_values('frame_n').reset_index(drop=True)
    frame_pipeline['nearest_color'] = frame_pipeline['mean_frame_score'].apply(lambda score: min(COLOR_TO_SCORE, key=lambda color: abs(score - COLOR_TO_SCORE[color])))
    frame_pipeline['confidence_half_band'] = frame_pipeline['nearest_color'].map(CONFIDENCE_BAND)
    frame_pipeline['lower_band'] = frame_pipeline['mean_frame_score'] - frame_pipeline['confidence_half_band']
    frame_pipeline['upper_band'] = frame_pipeline['mean_frame_score'] + frame_pipeline['confidence_half_band']
    return long_df, trial_matrix, frame_pipeline

def build_window_pipeline(frame_pipeline):
    max_frame = int(frame_pipeline['frame_n'].max())
    starts = list(np.arange(0, max_frame, STEP_SIZE))
    rows = []
    for start in starts:
        end = min(start + WINDOW_SIZE, max_frame)
        if end <= start: continue
        start_frame = max(1, int(round(start)) + 1)
        end_frame = int(round(end))
        chunk = frame_pipeline[(frame_pipeline['frame_n'] >= start_frame) & (frame_pipeline['frame_n'] <= end_frame)]
        if chunk.empty: continue
        score = chunk['mean_frame_score'].mean()
        color = min(COLOR_TO_SCORE, key=lambda item: abs(score - COLOR_TO_SCORE[item]))
        band = CONFIDENCE_BAND[color]
        rows.append({'window_id': len(rows), 'start_time_s': float(start), 'end_time_s': float(end), 'window_duration_s': float(end - start), 'start_frame': start_frame, 'end_frame': end_frame, 'window_size': WINDOW_SIZE, 'step_size': STEP_SIZE, 'pipeline_score': score, 'nearest_color': color, 'confidence_half_band': band, 'lower_band': score - band, 'upper_band': score + band})
    return pd.DataFrame(rows)

def run_pipeline(file_paths):
    long_df, trial_matrix, frame_pipeline = aggregate_frames(file_paths)
    window_pipeline = build_window_pipeline(frame_pipeline)
    return long_df, trial_matrix, frame_pipeline, window_pipeline


In [ ]:
long_df, trial_matrix, frame_pipeline, window_pipeline = run_pipeline(INPUT_FILES)
fig, ax = plt.subplots(figsize=(12, 6))
x = window_pipeline['window_id'].to_numpy(dtype=int)
y = window_pipeline['pipeline_score'].to_numpy(dtype=float)
lower = window_pipeline['lower_band'].to_numpy(dtype=float)
upper = window_pipeline['upper_band'].to_numpy(dtype=float)

# Fasce cromatiche dello sfondo: soglie richieste 0-24.5, 24.5-50, 50-74.5, 74.5-100.
ax.axhspan(0, 24.5, facecolor='#2e7d32', alpha=0.12, zorder=0)
ax.axhspan(24.5, 50, facecolor='#f9a825', alpha=0.12, zorder=0)
ax.axhspan(50, 74.5, facecolor='#ef6c00', alpha=0.12, zorder=0)
ax.axhspan(74.5, 100, facecolor='#c62828', alpha=0.12, zorder=0)

ax.plot(x, y, marker='o', linewidth=2, color='#1f77b4', label='Pipeline totale', zorder=3)
ax.fill_between(x, lower, upper, color='#1f77b4', alpha=0.18, label='Bande di confidenza', zorder=2)
ax.set_xlim(x.min() - 0.5, x.max() + 0.5)
ax.set_ylim(0, 100)
ax.set_xticks(x)
ax.set_xticklabels([str(value) for value in x])
ax.set_xlabel('Numero finestra temporale')
ax.set_ylabel('Punteggio medio finestra')
ax.set_title('Pipeline di punteggio del video')
ax.grid(True, axis='y', alpha=0.35)
ax.legend(loc='upper left')
plt.tight_layout()
plot_path = OUTPUT_DIR / 'pipeline_plot.png'
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Grafico salvato in: {plot_path}')
print(f'Numero frame rilevati: {int(frame_pipeline.frame_n.max())}')
print(f'Numero finestre generate: {len(window_pipeline)}')


In [ ]:
# EXPORT EXCEL
excel_path = OUTPUT_DIR / 'pipeline_scoring_output.xlsx'
trial_matrix_export = trial_matrix.copy()
trial_matrix_export.columns = [f'frame_{int(column)}' for column in trial_matrix_export.columns]
trial_matrix_export = trial_matrix_export.reset_index()
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    trial_matrix_export.to_excel(writer, sheet_name='votazioni_frame', index=False)
    frame_pipeline.to_excel(writer, sheet_name='media_frame', index=False)
    window_pipeline.to_excel(writer, sheet_name='pipeline_totale', index=False)
    long_df.to_excel(writer, sheet_name='dati_long', index=False)
wb = load_workbook(excel_path)
COLOR_HEX = {'verde': 'C6EFCE', 'giallo': 'FFF2CC', 'arancione': 'F4B183', 'rosso': 'F4CCCC'}
SCORE_HEX = {12.0: 'C6EFCE', 37.0: 'FFF2CC', 62.0: 'F4B183', 87.5: 'F4CCCC'}
def header_map(ws): return {ws.cell(row=1, column=col).value: col for col in range(1, ws.max_column + 1)}
def style_header(ws):
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center')
def fill_row_by_color(ws, row, color):
    if pd.isna(color): return
    color = str(color).strip().lower()
    if color not in COLOR_HEX: return
    fill = PatternFill(fill_type='solid', fgColor=COLOR_HEX[color])
    for col in range(1, ws.max_column + 1): ws.cell(row=row, column=col).fill = fill

ws = wb['votazioni_frame']
style_header(ws)
for row in range(2, ws.max_row + 1):
    for col in range(2, ws.max_column + 1):
        value = ws.cell(row=row, column=col).value
        try: numeric_value = float(value)
        except (TypeError, ValueError): numeric_value = None
        if numeric_value in SCORE_HEX: ws.cell(row=row, column=col).fill = PatternFill(fill_type='solid', fgColor=SCORE_HEX[numeric_value])

ws = wb['media_frame']
style_header(ws)
mapping = header_map(ws)
for row in range(2, ws.max_row + 1): fill_row_by_color(ws, row, ws.cell(row, mapping['nearest_color']).value)

ws = wb['pipeline_totale']
style_header(ws)
mapping = header_map(ws)
for row in range(2, ws.max_row + 1): fill_row_by_color(ws, row, ws.cell(row, mapping['nearest_color']).value)
if not window_pipeline.empty:
    chart = LineChart()
    chart.title = 'Pipeline totale con bande di confidenza'
    chart.y_axis.title = 'Score'
    chart.x_axis.title = 'Numero finestra temporale'
    chart.height = 9
    chart.width = 18
    for column in ['pipeline_score', 'lower_band', 'upper_band']:
        chart.add_data(Reference(ws, min_col=mapping[column], min_row=1, max_row=ws.max_row), titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=mapping['window_id'], min_row=2, max_row=ws.max_row))
    for series, color in zip(chart.series, ['1F77B4', '7F8C8D', '7F8C8D']): series.graphicalProperties = GraphicalProperties(ln=LineProperties(solidFill=color))
    ws.add_chart(chart, 'L4')

ws = wb['dati_long']
style_header(ws)
mapping = header_map(ws)
for row in range(2, ws.max_row + 1): fill_row_by_color(ws, row, ws.cell(row, mapping['traffic_light']).value)
wb.save(excel_path)
print(f'File Excel salvato in: {excel_path}')
